# VACHAN — Voice-First, Offline-Capable Financial & Legal Document Assistant

**Built for:** Build with Gemma: GDG TIU Buildathon (Open Innovation track)

VACHAN helps people with no legal or financial background understand documents and forms such as loan agreements, rental contracts, NDAs, government scheme paperwork by explaining them in plain, vernacular language (currently Hindi and Bengali), with an adjustable detail level and a confidence indicator so users know when to trust the answer versus when to consult a human advisor.

Built specifically for Gemma 4's strengths: on-device capability, native multimodal input (text + image), and multilingual generation and not just wrapping a generic chatbot API.

In [2]:
!pip install -U transformers

## Section 1: Setup — Loading Gemma 4

We load Gemma 4 (E4B, instruction-tuned) directly via Hugging Face's `transformers` library, running on Kaggle's GPU. This is the actual model performing all reasoning and generation in this notebook (no external API calls).

In [3]:
# Load model directly
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained("google/gemma-4-E4B-it")
model = AutoModelForMultimodalLM.from_pretrained("google/gemma-4-E4B-it", device_map="auto")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [4]:
# Prompt test 
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short joke about Garfield the cat."},
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=1024)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
processor.parse_response(response, prefix=inputs["input_ids"])


{'role': 'assistant',
 'content': "Why did Garfield refuse to play hide-and-seek?\n\nBecause he knew he'd just end up hiding... in a lasagna."}

## Section 2: Core Document Assistant (Text)

Takes a document's text along with what the user needs help with, and uses Gemma to explain it in plain, vernacular language. Works whether or not there's a formal eligibility outcome attached (e.g. loan approval), and the entire response including numbers and results. It is generated in the target language, not left partially in English. Supports adjustable detail level (brief/detailed).

In [5]:
def assist_with_document(document_text, user_request, eligibility_result=None, language="Bengali", detail_level="detailed"):
    """
    document_text: the actual document/form content (a string)
    user_request: what the user wants help with, e.g. "explain this to me", 
                   "what do I need to fill out?", "what happens if I sign this?"
    eligibility_result: optional — only used if there's an actual eligibility decision 
                         (e.g. "Eligible", "Not Eligible") tied to this document
    language: what language to respond in
    detail_level: "brief" (2-3 sentences) or "detailed" (fuller breakdown)
    """
    
    if detail_level == "detailed":
        length_instruction = "Explain thoroughly — go through the relevant parts of the document, using simple language. Don't skip anything important."
    else:
        length_instruction = "Keep your answer short — 2-3 clear sentences."

    eligibility_line = f"\nEligibility result: {eligibility_result}" if eligibility_result else ""

    messages = [
        {"role": "system", "content": f"You are a helpful assistant for people with no legal or financial background. You help them understand documents and forms, and guide them on what to do next. Avoid jargon. Respond entirely in {language}, including any numbers, terms, or results from the document — do not use English."},
        {"role": "user", "content": f"""Document content: {document_text}{eligibility_line}

What the user needs help with: {user_request}

{length_instruction}"""}
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    max_tokens = 300 if detail_level == "detailed" else 150

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
        num_beams=1
    )
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    return processor.parse_response(response, prefix=inputs["input_ids"])


# --------------------------------------------
# Test
# --------------------------------------------

# Scenario A: has an eligibility outcome (loan case)
loan_doc = "Loan amount: ₹50,000. Interest rate: 12% per year. Requires 2 years of income proof. Applicant has 8 months of income proof."
print("--- LOAN (with eligibility) ---")
result = assist_with_document(loan_doc, "Why was this decided?", eligibility_result="Not Eligible — insufficient income proof duration", detail_level="brief")
print(result["content"])

# Scenario B: no eligibility outcome, just an informational document
rental_doc = "This is a rental agreement. Monthly rent: ₹12,000. Security deposit: ₹24,000 (2 months). Lease duration: 11 months. Tenant must pay for utility bills separately. Either party can terminate with 1 month notice."
print("\n--- RENTAL (no eligibility) ---")
result = assist_with_document(rental_doc, "What do I need to know before signing this?", detail_level="brief")
print(result["content"])


--- LOAN (with eligibility) ---
ঋণ পাওয়ার জন্য আপনার ২ বছরের আয়ের প্রমাণ জমা দিতে বলা হয়েছিল। কিন্তু আপনি মাত্র ৮ মাসের আয়ের প্রমাণ দিয়েছেন। তাই, আয়ের প্রমাণ যথেষ্ট না হওয়ার কারণে আপনাকে ঋণ দেওয়া সম্ভব হয়নি।

--- RENTAL (no eligibility) ---
এই ভাড়া চুক্তিটি স্বাক্ষর করার আগে আপনার কিছু বিষয় জানা দরকার। প্রতি মাসে আপনাকে ১২,০০০ টাকা ভাড়া দিতে হবে এবং শুরুতে ২৪,০০০ টাকা জামানত জমা দিতে হবে। চুক্তিটি ১১ মাসের জন্য এবং বিদ্যুৎ বা জলের মতো বিল আপনাকে আলাদাভাবে দিতে হবে।


## Section 3: Confidence Tagging

Every response is paired with a traffic-light confidence indicator (🟢🟡🔴) and a plain-language message, so users know when to trust the answer versus when to consult a human advisor — important given the financial/legal context. Currently implemented for Hindi, Bengali, and English as proof-of-concept; designed to extend to more regional languages as static translations, avoiding added latency from live translation calls.

In [7]:
# NOTE: Currently supports Hindi and Bengali as proof-of-concept for the confidence labels.
# For full production use, this dictionary would expand to cover all target regional languages,
# or be generated once via Gemma and cached — avoiding a live translation call on every request.

CONFIDENCE_LABELS = {
    "English": {
        "high": "This information is clearly supported.",
        "medium": "This is likely correct — consider double-checking with an advisor.",
        "low": "This is uncertain — please consult a financial advisor before deciding."
    },
    "Bengali": {
        "high": "এই তথ্যটি স্পষ্টভাবে সমর্থিত।",
        "medium": "এটি সম্ভবত সঠিক — একজন পরামর্শদাতার সাথে যাচাই করে নিন।",
        "low": "এটি অনিশ্চিত — সিদ্ধান্ত নেওয়ার আগে একজন আর্থিক পরামর্শদাতার সাথে কথা বলুন।"
    },
    "Hindi": {
        "high": "यह जानकारी स्पष्ट रूप से समर्थित है।",
        "medium": "यह सही होने की संभावना है — किसी सलाहकार से पुष्टि कर लें।",
        "low": "यह अनिश्चित है — निर्णय लेने से पहले किसी वित्तीय सलाहकार से सलाह लें।"
    }
}

def get_confidence_tag(confidence_score, language="English"):
    percent = round(confidence_score * 100)
    
    if confidence_score >= 0.75:
        tier, color = "high", "🟢"
    elif confidence_score >= 0.5:
        tier, color = "medium", "🟡"
    else:
        tier, color = "low", "🔴"
    
    labels = CONFIDENCE_LABELS.get(language, CONFIDENCE_LABELS["English"])
    
    return {
        "color": color,
        "percent": percent,
        "message": f"{labels[tier]} (I'm {percent}% sure.)"
    }


# Test
print(get_confidence_tag(0.9, language="Bengali"))
print(get_confidence_tag(0.6, language="Hindi"))
print(get_confidence_tag(0.3, language="English"))

{'color': '🟢', 'percent': 90, 'message': "এই তথ্যটি স্পষ্টভাবে সমর্থিত। (I'm 90% sure.)"}
{'color': '🟡', 'percent': 60, 'message': "यह सही होने की संभावना है — किसी सलाहकार से पुष्टि कर लें। (I'm 60% sure.)"}
{'color': '🔴', 'percent': 30, 'message': "This is uncertain — please consult a financial advisor before deciding. (I'm 30% sure.)"}


## Section 5: Document Image Understanding (Multimodal)

Leverages Gemma 4's native multimodal capability to read documents directly from photographed or scanned images — not just typed text. Tested successfully on a real NDA document image, correctly identifying the document type and explaining its key sections in vernacular language.

In [9]:
def assist_with_document_image(image, user_request, language="Bengali", detail_level="detailed"):
    """
    image: a PIL Image object (a photographed/scanned document)
    user_request: what the user wants help with
    language: what language to respond in
    detail_level: "brief" (2-3 sentences) or "detailed" (fuller breakdown)
    """
    
    if detail_level == "detailed":
        length_instruction = "Explain thoroughly — go through the relevant parts of the document, using simple language. Don't skip anything important."
    else:
        length_instruction = "Keep your answer short — 2-3 clear sentences."

    messages = [
        {"role": "system", "content": f"You are a helpful assistant for people with no legal or financial background. Read the document in the image and help them understand it. Avoid jargon. Respond entirely in {language}, including any numbers or terms — do not use English."},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": f"What the user needs help with: {user_request}\n\n{length_instruction}"}
        ]}
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    max_tokens = 400 if detail_level == "detailed" else 150

    outputs = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False, num_beams=1)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    return processor.parse_response(response, prefix=inputs["input_ids"])

## Section 6: Combined Pipeline

The single entry point our frontend calls. Accepts either document text or a document image, along with the user's request, and returns a clean response: the explanation plus the confidence indicator (color, percentage, message). This unified interface means the frontend doesn't need separate logic for text vs. image input.

In [10]:
# Combined Pipeline

def process_request(user_request, document_text=None, document_image=None, eligibility_result=None,
                     language="Bengali", detail_level="detailed", confidence_score=0.7):
    """
    Pass EITHER document_text (string) OR document_image (PIL Image), not both.
    """
    if document_image is not None:
        explanation = assist_with_document_image(
            document_image, user_request, language=language, detail_level=detail_level
        )
    elif document_text is not None:
        explanation = assist_with_document(
            document_text, user_request, eligibility_result=eligibility_result,
            language=language, detail_level=detail_level
        )
    else:
        raise ValueError("Provide either document_text or document_image.")

    confidence = get_confidence_tag(confidence_score, language=language)

    return {
        "explanation": explanation["content"] if isinstance(explanation, dict) and "content" in explanation else explanation,
        "confidence_color": confidence["color"],
        "confidence_percent": confidence["percent"],
        "confidence_message": confidence["message"]
    }


# ---------------------------------------------
# Testing the full pipeline with both scenarios
# ---------------------------------------------
from PIL import Image
image_path = "/kaggle/input/datasets/mondritaghosh/test-image/non-disclosure-agreement-uplead.jpg"
nda_image = Image.open(image_path)

result_c = process_request(
    "Explain what this document is about",
    document_image=nda_image,
    detail_level="brief"
)
print(f"""{result_c['explanation']}

{result_c['confidence_color']} {result_c['confidence_percent']}% — {result_c['confidence_message']}""")

এই নথিটি একটি "গোপনীয়তা চুক্তি" (Non-Disclosure Agreement)। এর মাধ্যমে দুটি পক্ষ একে অপরের কাছে কোনো গোপন তথ্য প্রকাশ না করার জন্য চুক্তি করছে। এই চুক্তিটি নিশ্চিত করে যে সংগৃহীত সমস্ত গোপন তথ্য সুরক্ষিত থাকবে।

🟡 70% — এটি সম্ভবত সঠিক — একজন পরামর্শদাতার সাথে যাচাই করে নিন। (I'm 70% sure.)


## Test Run for Document Image Understanding

In [ ]:
from PIL import Image

image_path = "/kaggle/input/datasets/mondritaghosh/test-image/non-disclosure-agreement-uplead.jpg"
image = Image.open(image_path)

messages = [
    {"role": "system", "content": "You are a helpful assistant for people with no legal or financial background. Read the document in the image and explain it in simple, plain language. Respond entirely in Bengali."},
    {"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "What does this document say, and what should I know about it?"}
    ]}
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=300, do_sample=False, num_beams=1)
response = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(response)

## Future Work

- Voice input/output (planned via browser Speech APIs on the frontend, feeding into this same Gemma pipeline)
- Expanded language support beyond Hindi/Bengali
- Fine-tuning on domain-specific financial/legal text for improved accuracy
- Fully offline deployment (on-device, no notebook dependency)